# 08 — Mega-Cap Exclusion

Test whether momentum performance is mechanically distorted by the direct presence of the largest companies in the formation universe.

## Methodology locked before results
- Mega-cap ranking source: broad market-state universe.
- Company identifier: PERMCO.
- Primary exclusion: top 10 companies each formation month.
- Robustness: top 5 and top 20.
- Momentum signal definition is unchanged.
- Primary comparison: ExTop10 VW Q5−Q1 minus baseline VW Q5−Q1.
- HAC6 primary / HAC12 robustness.


In [ ]:
from pathlib import Path
import sys

ROOT = Path('..').resolve() if Path.cwd().name == 'notebooks' else Path('.').resolve()
sys.path.append(str(ROOT))

from src.clean_monthly import CleaningConfig, build_clean_monthly_panel, build_monthly_history_panel, build_market_state_panel
from src.signals import attach_momentum_to_formation_universe, assign_signal_quintiles
from src.portfolio import monthly_rank_ic, quintile_returns, quintile_return_panel
from src.mega_cap import exclude_top_permco, compare_spreads, delta_hac_summary


## 1. Rebuild validated baseline inputs


In [ ]:
RAW = ROOT / 'data/raw/monthly_stock_common_equity_2000_2025_prod_v1.csv.gz'
clean, _ = build_clean_monthly_panel(RAW, CleaningConfig())
history = build_monthly_history_panel(RAW)
market_state = build_market_state_panel(RAW)

mom_base = attach_momentum_to_formation_universe(clean, history)
mom_base = assign_signal_quintiles(mom_base)
qret_base = quintile_returns(mom_base)
vw_base = quintile_return_panel(qret_base, return_col='VW_Return')
ew_base = quintile_return_panel(qret_base, return_col='EW_Return')
ic_base = monthly_rank_ic(mom_base)


## 2. Primary: exclude top 10 PERMCO companies


In [ ]:
clean_ex10 = exclude_top_permco(clean, market_state, n=10)
mom_ex10 = attach_momentum_to_formation_universe(clean_ex10, history)
mom_ex10 = assign_signal_quintiles(mom_ex10)
qret_ex10 = quintile_returns(mom_ex10)
vw_ex10 = quintile_return_panel(qret_ex10, return_col='VW_Return')
ew_ex10 = quintile_return_panel(qret_ex10, return_col='EW_Return')
ic_ex10 = monthly_rank_ic(mom_ex10)

print(f'Baseline formation rows: {len(clean):,}')
print(f'ExTop10 formation rows:  {len(clean_ex10):,}')


## 3. Primary spread comparison


In [ ]:
cmp10 = compare_spreads(vw_base, vw_ex10, alt_label='ExTop10')
cmp10.tail()


In [ ]:
delta10_hac = delta_hac_summary(cmp10, alt_label='ExTop10')
delta10_hac


## 4. Economic-size comparison


In [ ]:
summary10 = {
    'Baseline_VW_mean': vw_base['Q5_minus_Q1'].mean(),
    'ExTop10_VW_mean': vw_ex10['Q5_minus_Q1'].mean(),
    'Delta_mean': cmp10['Delta_ExTop10_Minus_Baseline'].mean(),
    'Baseline_EW_mean': ew_base['Q5_minus_Q1'].mean(),
    'ExTop10_EW_mean': ew_ex10['Q5_minus_Q1'].mean(),
    'Baseline_RankIC_mean': ic_base['RankIC'].mean(),
    'ExTop10_RankIC_mean': ic_ex10['RankIC'].mean(),
}
summary10


## 5. Robustness: exclude top 5 and top 20


In [ ]:
results = {}
for n in [5, 20]:
    clean_alt = exclude_top_permco(clean, market_state, n=n)
    mom_alt = attach_momentum_to_formation_universe(clean_alt, history)
    mom_alt = assign_signal_quintiles(mom_alt)
    qret_alt = quintile_returns(mom_alt)
    vw_alt = quintile_return_panel(qret_alt, return_col='VW_Return')
    cmp = compare_spreads(vw_base, vw_alt, alt_label=f'ExTop{n}')
    results[n] = {
        'MeanSpread': vw_alt['Q5_minus_Q1'].mean(),
        'MeanDelta': cmp[f'Delta_ExTop{n}_Minus_Baseline'].mean(),
        'HAC': delta_hac_summary(cmp, alt_label=f'ExTop{n}')
    }

results[5]['MeanSpread'], results[5]['MeanDelta'], results[5]['HAC']


In [ ]:
results[20]['MeanSpread'], results[20]['MeanDelta'], results[20]['HAC']


## 6. Composition diagnostic


In [ ]:
base_counts = mom_base.groupby(['MthCalDt','SignalQuintile'], observed=True).size().unstack()
ex10_counts = mom_ex10.groupby(['MthCalDt','SignalQuintile'], observed=True).size().unstack()
count_diff = (ex10_counts - base_counts).tail()
count_diff


## Interpretation discipline

The primary question is whether excluding the top 10 companies changes the VW momentum spread relative to the baseline. Compare the delta directly; do not infer an effect merely because one standalone spread is significant and another is not.
